## 0. 파일 확인

In [3]:
import pandas as pd

sub = pd.read_csv("../data/clean/종속기업_정리.csv", dtype=str)
corp = pd.read_csv("../data/clean/기업개요_최종.csv", dtype=str)

def norm(s):
    for w in ["(주)", "주식회사", "㈜", "(유)", "유한회사", " "]:
        s = s.replace(w, "")
    return s.upper()

corp["name_norm"] = corp["corpNm"].fillna("").apply(norm)

## 1. 종속기업이면서 모회사인 것
- 라벨 2개 생성 방지

In [4]:
parents = set(sub["crno"])   # 종속기업을 가진 회사들

m = sub.merge(corp[["crno", "name_norm"]].rename(columns={"crno": "b_crno"}),
              on="name_norm")
chain = m[m["b_crno"].isin(parents)]

print("B 후보:", chain["b_crno"].nunique(), "개")
print(chain[["crno", "sbrdEnpNm", "b_crno"]].head())

B 후보: 65 개
             crno   sbrdEnpNm         b_crno
0   1101110017867   한솔피엔에스(주)  1101110175970
1   1101110017867  한솔로지스틱스(주)  1101110150659
17  1101110085450     현대카드(주)  1101110377203
21  1101110108484     아시아나항공㈜  1101110562804
22  1101110108484       한국공항㈜  1101110003692


In [9]:
nodes = pd.read_json("../data/clean/기업관계_노드.jsonl", lines=True, dtype=False)
triples = pd.read_json("../data/clean/기업관계_트리플.jsonl", lines=True, dtype=False)

print(nodes.shape, nodes.columns.tolist())
print(nodes.head(3))
print(triples.shape, triples.columns.tolist())
print(triples.head(3))

(10350, 3) ['id', 'type', 'properties']
                             id           type  \
0  parent_company:1101110000086  ParentCompany   
1  parent_company:1101110002694  ParentCompany   
2  parent_company:1101110002818  ParentCompany   

                                          properties  
0  {'crno': '1101110000086', 'name': '롯데쇼핑(주)', '...  
1  {'crno': '1101110002694', 'name': '지에스건설(주)', ...  
2  {'crno': '1101110002818', 'name': '한국제지(주)', '...  
(19329, 8) ['subject', 'subject_type', 'relation', 'object', 'object_type', 'source_case', 'source_row', 'evidence']
                        subject   subject_type         relation  \
0  parent_company:1101110000086  ParentCompany  AFFILIATED_WITH   
1  parent_company:1101110000086  ParentCompany       LOCATED_IN   
2  parent_company:1101110014764  ParentCompany       LOCATED_IN   

                         object    object_type source_case  source_row  \
0  parent_company:1101110014764  ParentCompany           1           0   
1    

In [10]:
print("빈 id:", nodes["id"].isna().sum())
print("겹치는 id:", nodes["id"].duplicated().sum())

빈 id: 0
겹치는 id: 0


In [11]:
ids = set(nodes["id"])
print("출발 없음:", (~triples["subject"].isin(ids)).sum())
print("도착 없음:", (~triples["object"].isin(ids)).sum())

출발 없음: 0
도착 없음: 0


In [12]:
nodes["name"] = nodes["properties"].apply(lambda p: p.get("name"))
dup = nodes[nodes["name"].duplicated(keep=False)].sort_values("name")

print("이름 겹치는 노드:", len(dup))
print(dup[["id", "type", "name"]].head(10))
print(nodes[nodes["name"].str.contains("아시아나", na=False)][["id", "type", "name"]])

이름 겹치는 노드: 648
                                                     id               type  \
7570  subsidiary:name:문경가온태양광발전|address:경상북도문경시신기로20유곡동  SubsidiaryCompany   
7968     subsidiary:name:문경가온태양광발전|address:경상북도문경시신기로20  SubsidiaryCompany   
7969  subsidiary:name:전주가온태양광발전|address:전라북도전주시덕진구서귀로77  SubsidiaryCompany   
7569  subsidiary:name:전주가온태양광발전|address:전라북도전주시덕진구서귀...  SubsidiaryCompany   
1895           subsidiary:name:가하이엠씨|address:경남거제시장평로12  SubsidiaryCompany   
6249         subsidiary:name:가하이엠씨|address:서울시구로구경인로662  SubsidiaryCompany   
1894        subsidiary:name:가하컨설팅|address:서울시성동구광나루로146  SubsidiaryCompany   
6247         subsidiary:name:가하컨설팅|address:서울시구로구경인로662  SubsidiaryCompany   
410                        parent_company:1101110404361      ParentCompany   
5713            subsidiary:name:동성화인텍|address:경기도안성시미양면  SubsidiaryCompany   

              name  
7570  (유)문경가온태양광발전  
7968  (유)문경가온태양광발전  
7969  (유)전주가온태양광발전  
7569  (유)전주가온태양광발전  
1895      (주)가하이엠씨  
